# GeoLibre basics

The `geolibre` package embeds the **full** GeoLibre app in a notebook cell as an
anywidget. Not a stripped-down map widget — the real menus, panels, and
Processing toolbox, driven by a leafmap-style API.

State syncs *both ways* through a single `.geolibre.json` project: data you add
from Python shows up in the UI, and pans/zooms/edits you make in the UI read
back into Python.

Run `pixi run lab` from the project root, then work through this top to bottom.


In [ ]:
import geolibre
from geolibre import Map

print("geolibre", geolibre.__version__)

## 1. A map

`layout=` controls how much UI you get: `"embed"` (compact), `"full"` (the
desktop UI, all menus), or `"maponly"`. Use `"full"` while you're learning —
that's how you reach Processing → Raster → Spectral Index later.


In [ ]:
m = Map(
    center=(-120.5, 38.8),   # [lng, lat] -- Sierra Nevada foothills
    zoom=8,
    basemap="dark",
    height="700px",
    layout="full",
    theme="dark",
)
m

## 2. Add some data

Note the argument order everywhere in this API is **(lng, lat)**, not (lat, lng).


In [ ]:
m.add_tile_layer(
    "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="(c) OpenStreetMap contributors",
)

m.add_marker(-120.09, 38.94, properties={"name": "Placerville"})
m.add_marker(-121.06, 39.22, properties={"name": "Nevada City"})

## 3. Two-way sync

Pan and zoom the map above, *then* run this cell. `to_project()` returns the
live project dict, so `mapView.center` reflects whatever you just did with the
mouse. This is the part that makes the widget feel different from leafmap.


In [ ]:
proj = m.to_project()
print("center:", proj["mapView"]["center"])
print("layers:", [layer["name"] for layer in proj["layers"]])

### Blocking queries

The interactive query methods round-trip to the running app and **block the
kernel until it replies** (via `jupyter_ui_poll`). So the map must already be
displayed in an earlier cell before you call these. Pass `timeout=` for slow
ones.


In [ ]:
print(m.get_center())
print(m.get_bounds())

# Like clicking the map -- returns rendered features at a point
m.identify(-120.09, 38.94)

## 4. Layer handles

`m.layers` gives you handles you can read and mutate directly.


In [ ]:
for layer in m.layers:
    print(f"{layer.name:24s} visible={layer.visible} opacity={layer.opacity}")

# Mutating a handle pushes straight to the app
m.layers[0].opacity = 0.6

## 5. Processing from Python

`list_algorithms()` enumerates what's available with parameter specs.
**Run this and read the output carefully** — it's how you'll find the exact
algorithm id for NDVI in notebook 02, which I'd rather you confirm than take
on faith from me.


In [ ]:
algos = m.list_algorithms()
print(len(algos), "algorithms")

# Look for the spectral index tools
for a in algos:
    if any(k in str(a).lower() for k in ("ndvi", "ndwi", "evi", "spectral", "index")):
        print(a)

In [ ]:
# Once you know the id and its parameters:
# result = m.run_algorithm("buffer", {"layer": layer_id, "distance": 1000}, timeout=300)
# result -> {"logs": [...], "resultLayerIds": [...]}

## 6. Save / share

`.geolibre.json` is fully interchangeable with the desktop and web apps — save
here, open on the phone.

Credentials (API keys, tokens, auth headers) are **redacted by default** on
`to_project()`, `save_project()`, and `to_html()`. Pass `keep_credentials=True`
only for a trusted local file.


In [ ]:
m.save_project("../projects/foothills.geolibre.json")

# Standalone HTML you can drop straight onto brooksgroves.com
m.to_html("../projects/foothills.html", title="Sierra Foothills")

### Publishing note

The web app reads a public project URL via the `url` query parameter, and takes
layout flags for embedding:

```
https://web.geolibre.app/?url=https://<your-host>/foothills.geolibre.json&layout=viewer
```

`layout=viewer` is read-only but still explorable — keeps the Layers list,
basemaps, and identify, hides everything that would change the project.
`&maponly` strips all chrome. `&layout=compact&panels=none` is the middle
ground for a narrow embed. There's also a typed `@geolibre/embed` client and a
versioned `postMessage` API if you want the host page to drive the map.
